# Modular Evaluation and Enhancement of VillageNet for Unsupervised Clustering
**Author:** Arthur Santos (DCOMP - UFSJ)  
**Advisor:** Jesuliana Nascimento Ulysses

This notebook implements the modular architecture of the **VillageNet** algorithm (Ballal et al., 2026) to evaluate structural modifications in **Steps 1, 3, and 4**:
- **Experiment 1 (Step 4 - Communities):** Replacing WLCF on the Inter-Village Graph with *Leiden*, *Spectral Clustering*, and *Infomap*.
- **Experiment 2 (Steps 1 and 3 - Villages and Boundary):** Replacing standard K-Means with *K-Means++*, *MiniBatch K-Means*, and *K-Medoids*, along with continuous distance-based weighting on edges $A_{UV}$.

## 1. Importing Libraries

In [1]:
from infomap import Infomap
import igraph as ig
import leidenalg
import numpy as np
import os
import pandas as pd
from scipy.sparse.csgraph import laplacian
from scipy.spatial.distance import cdist
from sklearn.cluster import KMeans, MiniBatchKMeans, SpectralClustering
from sklearn.datasets import load_breast_cancer, load_digits, load_wine
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
from sklearn.preprocessing import StandardScaler
import time

## 2. Helper Functions (K-Medoids and Spectral Estimation of $k$)
- `_pam_kmedoids`: Lightweight K-Medoids (PAM) implementation to test robustness against *outliers* in Step 1.
- `_estimate_k_eigengap`: Automatically estimates the number of clusters $k$ via the largest eigenvalue gap (*eigengap*) of the normalized Laplacian matrix of the Inter-Village Graph, keeping *Spectral Clustering* 100% unsupervised.

In [2]:
def _pam_kmedoids(X, n_clusters, max_iter=100, random_state=42):
    """Lightweight K-Medoids implementation (Step 1 robust to outliers) using L2 distance."""
    rng = np.random.default_rng(random_state)
    N = X.shape[0]
    medoid_indices = rng.choice(N, size=n_clusters, replace=False)
    medoids = X[medoid_indices].copy()

    for _ in range(max_iter):
        dists = cdist(X, medoids, metric='euclidean')
        labels = np.argmin(dists, axis=1)
        new_medoids = medoids.copy()
        for k in range(n_clusters):
            cluster_pts = X[labels == k]
            if len(cluster_pts) > 0:
                intra_dists = cdist(
                    cluster_pts, cluster_pts, metric='euclidean'
                ).sum(axis=1)
                new_medoids[k] = cluster_pts[np.argmin(intra_dists)]
        if np.allclose(medoids, new_medoids):
            break
        medoids = new_medoids

    dists = cdist(X, medoids, metric='euclidean')
    labels = np.argmin(dists, axis=1)
    return labels, medoids


def _estimate_k_eigengap(adjacency_matrix, max_k=25):
    """Automatically estimates the number of clusters k via the Laplacian Eigengap."""
    n_nodes = adjacency_matrix.shape[0]
    max_k = min(max_k, n_nodes - 1)
    if max_k < 2:
        return 1

    L = laplacian(adjacency_matrix, normed=True)
    eigenvalues = np.linalg.eigvalsh(L)
    eigenvalues = np.sort(np.real(eigenvalues))

    # Ignore the first eigenvalue (~0) and find the largest jump (eigengap)
    gaps = np.diff(eigenvalues[1 : max_k + 1])
    if len(gaps) == 0:
        return 2
    optimal_k = int(np.argmax(gaps) + 2)
    return optimal_k

## 3. `ModularVillageNet` Class (The 5 Pluggable Steps)
Implementation of the original equations from Ballal et al. (2026) with support for the proposed variations:
1. **Step 1 (`_step1_coarse_graining`):** Creates the $\nu$ villages (Voronoi cells) via `kmeans_random`, `kmeans++`, `minibatch`, or `kmedoids`.
2. **Steps 2 and 3 (`_step2_and_3_inter_village_graph`):** Computes the boundary projection $\mathcal{D}(\vec{X}_i, U)$ between villages, selects the $\eta$ nearest external points via `np.argpartition`, and builds the adjacency matrix $A_{UV}$ (`discrete` or `continuous`).
3. **Step 4 (`_step4_community_detection`):** Detects communities in the inter-village network (`leiden`, `spectral`, `infomap`, or `louvain_igraph`).
4. **Step 5 (`fit_predict`):** Merges data points belonging to villages in the same community.

In [3]:
class ModularVillageNet:

    def __init__(
        self,
        nu=None,
        eta=30,
        village_method='kmeans_random',
        weight_method='discrete',
        community_method='leiden',
        random_state=42,
    ):
        self.nu = nu
        self.eta = eta
        self.village_method = village_method
        self.weight_method = weight_method
        self.community_method = community_method
        self.random_state = random_state

        self.labels_ = np.array([], dtype=int)
        self.n_clusters_ = 0
        self.step_times_ = {}
        self.total_time_ = 0.0

    def _step1_coarse_graining(self, X, nu):
        if self.village_method == 'kmeans_random':
            km = KMeans(
                n_clusters=nu,
                init='random',
                n_init=1,
                random_state=self.random_state,
            )
            village_labels = km.fit_predict(X)
            centroids = km.cluster_centers_
        elif self.village_method == 'kmeans++':
            km = KMeans(
                n_clusters=nu,
                init='k-means++',
                n_init=10,
                random_state=self.random_state,
            )
            village_labels = km.fit_predict(X)
            centroids = km.cluster_centers_
        elif self.village_method == 'minibatch':
            mb = MiniBatchKMeans(
                n_clusters=nu,
                init='k-means++',
                n_init=3,
                batch_size=1024,
                random_state=self.random_state,
            )
            village_labels = mb.fit_predict(X)
            centroids = mb.cluster_centers_
        elif self.village_method == 'kmedoids':
            village_labels, centroids = _pam_kmedoids(
                X, n_clusters=nu, random_state=self.random_state
            )
        else:
            raise ValueError(
                f'Unknown village method: {self.village_method}'
            )

        return village_labels, centroids

    def _step2_and_3_inter_village_graph(
        self, X, village_labels, centroids, nu, eta
    ):
        A = np.zeros((nu, nu), dtype=np.float64)
        mu_Vi = centroids[village_labels]

        for u in range(nu):
            mask_outside_u = village_labels != u
            idx_outside = np.where(mask_outside_u)[0]
            if len(idx_outside) == 0:
                continue

            X_out = X[idx_outside]
            mu_V_out = mu_Vi[idx_outside]
            mu_U = centroids[u]

            diff_mu = mu_V_out - mu_U
            norm_diff = np.linalg.norm(diff_mu, axis=1, keepdims=True)
            norm_diff = np.where(norm_diff == 0, 1e-12, norm_diff)
            unit_vec = diff_mu / norm_diff

            midpoint = 0.5 * (mu_U + mu_V_out)
            D_Xi_U = np.sum((X_out - midpoint) * unit_vec, axis=1)
            D_Xi_U = np.abs(D_Xi_U)

            k_ext = min(eta, len(D_Xi_U) - 1)
            if k_ext <= 0:
                continue

            ext_local_idx = np.argpartition(D_Xi_U, k_ext)[:k_ext]
            ext_global_idx = idx_outside[ext_local_idx]
            ext_villages = village_labels[ext_global_idx]

            if self.weight_method == 'discrete':
                unique_v, counts = np.unique(ext_villages, return_counts=True)
                for v, c in zip(unique_v, counts):
                    A[u, v] += c
                    A[v, u] += c
            elif self.weight_method == 'continuous':
                ext_dists = D_Xi_U[ext_local_idx]
                sigma = np.median(ext_dists) + 1e-8
                weights = np.exp(-(ext_dists**2) / (2 * (sigma**2)))
                for v, w in zip(ext_villages, weights):
                    A[u, v] += w
                    A[v, u] += w
            else:
                raise ValueError(
                    f'Unknown weight method: {self.weight_method}'
                )

        np.fill_diagonal(A, 0.0)
        return A

    def _step4_community_detection(self, A, nu):
        if self.community_method in ['leiden', 'louvain_igraph']:
            sources, targets = np.nonzero(np.triu(A, k=1))
            weights = A[sources, targets].tolist()
            g = ig.Graph(
                n=nu,
                edges=list(zip(sources.tolist(), targets.tolist())),
                directed=False,
            )
            g.es['weight'] = weights

            if self.community_method == 'leiden':
                partition = leidenalg.find_partition(
                    g,
                    leidenalg.ModularityVertexPartition,
                    weights='weight',
                    seed=self.random_state,
                )
            else:
                partition = g.community_multilevel(weights='weight')

            village_communities = np.array(
                getattr(partition, 'membership', []), dtype=int
            )

        elif self.community_method == 'infomap':
            im = Infomap("--two-level --silent")
            sources, targets = np.nonzero(np.triu(A, k=1))
            for u, v in zip(sources, targets):
                im.add_link(int(u), int(v), float(A[u, v]))
            result = im.run()
            village_communities = np.arange(nu, dtype=int)
            for node in result.nodes(states=True):
                village_communities[node.node_id] = node.module_id
            _, village_communities = np.unique(
                village_communities, return_inverse=True
            )

        elif self.community_method == 'spectral':
            k_est = _estimate_k_eigengap(A)
            A_reg = A + 1e-6 * np.eye(nu)
            sc = SpectralClustering(
                n_clusters=k_est,
                affinity='precomputed',
                assign_labels='kmeans',
                random_state=self.random_state,
            )
            village_communities = sc.fit_predict(A_reg)
        else:
            raise ValueError(
                f'Unknown community method: {self.community_method}'
            )

        return village_communities

    def fit_predict(self, X):
        X = np.asarray(X, dtype=np.float64)
        N = X.shape[0]

        nu = self.nu if self.nu is not None else max(2, min(N // 10, 500))
        eta = min(self.eta, N - 1)

        t0 = time.perf_counter()

        t_s1 = time.perf_counter()
        village_labels, centroids = self._step1_coarse_graining(X, nu)
        self.step_times_['step1_villages'] = time.perf_counter() - t_s1

        t_s23 = time.perf_counter()
        A = self._step2_and_3_inter_village_graph(
            X, village_labels, centroids, nu, eta
        )
        self.step_times_['step2_3_graph'] = time.perf_counter() - t_s23

        t_s4 = time.perf_counter()
        village_communities = self._step4_community_detection(A, nu)
        self.step_times_['step4_community'] = time.perf_counter() - t_s4

        t_s5 = time.perf_counter()
        final_labels = village_communities[village_labels]
        _, final_labels = np.unique(final_labels, return_inverse=True)
        self.step_times_['step5_merging'] = time.perf_counter() - t_s5

        self.total_time_ = time.perf_counter() - t0
        self.labels_ = final_labels
        self.n_clusters_ = len(np.unique(final_labels))

        return self.labels_

## 4. Running Experiments 1 and 2 on Benchmark Datasets
We evaluate the variations on three public datasets used in the original paper (*Wine*, *Breast Cancer*, and *Pen Digits*), measuring:
- **External Metrics:** NMI, ARI, and cluster count error $|m - m^*|$.
- **Internal Metric:** Silhouette Coefficient.
- **Performance:** Execution time in seconds.

In [4]:
X_wine, y_wine = [np.asarray(a) for a in load_wine(return_X_y=True)]
X_bc, y_bc = [np.asarray(a) for a in load_breast_cancer(return_X_y=True)]
X_digits, y_digits = [np.asarray(a) for a in load_digits(return_X_y=True)]

datasets = {
    "Wine (m*=3)": (StandardScaler().fit_transform(X_wine), y_wine, 3),
    "Breast Cancer (m*=2)": (StandardScaler().fit_transform(X_bc), y_bc, 2),
    "Pen Digits (m*=10)": (
        StandardScaler().fit_transform(X_digits),
        y_digits,
        10,
    ),
}

configurations = [
    # Experiment 1: Varying community detection algorithm (Step 4)
    (
        "Exp1: KMeans-Random + Discrete + Louvain",
        "kmeans_random",
        "discrete",
        "louvain_igraph",
    ),
    (
        "Exp1: KMeans-Random + Discrete + Leiden",
        "kmeans_random",
        "discrete",
        "leiden",
    ),
    (
        "Exp1: KMeans-Random + Discrete + Spectral",
        "kmeans_random",
        "discrete",
        "spectral",
    ),
    (
        "Exp1: KMeans-Random + Discrete + Infomap",
        "kmeans_random",
        "discrete",
        "infomap",
    ),
    # Experiment 2: Varying Step 1 (Villages) and Step 3 (Edge Weights)
    (
        "Exp2: KMeans++ + Discrete + Leiden",
        "kmeans++",
        "discrete",
        "leiden",
    ),
    (
        "Exp2: MiniBatch + Discrete + Leiden",
        "minibatch",
        "discrete",
        "leiden",
    ),
    (
        "Exp2: KMedoids + Discrete + Leiden",
        "kmedoids",
        "discrete",
        "leiden",
    ),
    (
        "Exp2: KMeans++ + Continuous + Leiden",
        "kmeans++",
        "continuous",
        "leiden",
    ),
]

results = []

for ds_name, (X, y_true, m_star) in datasets.items():
    for name, v_meth, w_meth, c_meth in configurations:
        model = ModularVillageNet(
            nu=None,
            eta=30,
            village_method=v_meth,
            weight_method=w_meth,
            community_method=c_meth,
            random_state=42,
        )
        y_pred = model.fit_predict(X)

        nmi = normalized_mutual_info_score(y_true, y_pred)
        ari = adjusted_rand_score(y_true, y_pred)
        sil = (
            silhouette_score(X, y_pred) if model.n_clusters_ > 1 else np.nan
        )
        err_k = abs(model.n_clusters_ - m_star)

        results.append({
            "Dataset": ds_name,
            "Configuration": name,
            "NMI": round(nmi, 4),
            "ARI": round(ari, 4),
            "Silhouette": round(sil, 4) if not np.isnan(sil) else None,
            "m_inferred": model.n_clusters_,
            "Error_|m-m*|": err_k,
            "Time_s": round(model.total_time_, 4),
        })

df_results = pd.DataFrame(results)
os.makedirs("results", exist_ok=True)
df_results.to_csv("results/initial_benchmark.csv", index=False)
df_results

,Dataset,Configuration,NMI,ARI,Silhouette,m_inferred,Error_|m-m*|,Time_s
0,Wine (m*=3),Exp1: KMeans-Random + Discrete + Louvain,0.8920,0.9134,0.2815,3,0,0.0231
1,Wine (m*=3),Exp1: KMeans-Random + Discrete + Leiden,0.8920,0.9134,0.2815,3,0,0.0022
2,Wine (m*=3),Exp1: KMeans-Random + Discrete + Spectral,0.8920,0.9134,0.2815,3,0,0.0052
3,Wine (m*=3),Exp1: KMeans-Random + Discrete + Infomap,0.8920,0.9134,0.2815,3,0,0.0029
4,Wine (m*=3),Exp2: KMeans++ + Discrete + Leiden,0.7438,0.7603,0.2735,3,0,0.0095
5,Wine (m*=3),Exp2: MiniBatch + Discrete + Leiden,0.7402,0.7544,0.2649,3,0,0.0053
6,Wine (m*=3),Exp2: KMedoids + Discrete + Leiden,0.8396,0.8803,0.2793,3,0,0.0023
7,Wine (m*=3),Exp2: KMeans++ + Continuous + Leiden,0.7438,0.7603,0.2735,3,0,0.0134
8,Breast Cancer (m*=2),Exp1: KMeans-Random + Discrete + Louvain,0.4643,0.3961,0.0905,4,2,0.0091
9,Breast Cancer (m*=2),Exp1: KMeans-Random + Discrete + Leiden,0.4156,0.3296,0.1279,4,2,0.0215
